<a href="https://colab.research.google.com/github/RomanLuja/ML/blob/main/05_feature_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Заняття 5. Відбір ознак: як позбутися зайвого

**Блок 2. Дані як основа ML**

## 1. Навіщо відбирати ознаки

| Причина | Пояснення |
|---|---|
| **Шум** | випадкові ознаки модель може «запам'ятати» як закономірність — і помилятися на нових даних |
| **Пояснюваність** | модель з 8 ознаками легше пояснити бізнесу, ніж модель з 80 |
| **Вартість даних** | кожну ознаку треба збирати, зберігати, перевіряти щодня |
| **Витоки й дублікати** | їх треба прибрати незалежно від якості моделі |

### Три групи методів

1. **Ручні перевірки (здоровий глузд):** константи, ідентифікатори, дублікати, витоки.
2. **Фільтри (статистика):** кореляція з ціллю, `SelectKBest` — оцінка кожної ознаки окремо.
3. **Модельні:** модель сама вирішує, що важливо (`Lasso` обнуляє непотрібні коефіцієнти,
   дерева рахують важливість).

**Головне правило:** відбір теж «вчиться» на даних, тому робимо його **лише на train** і
перевіряємо результат на test.

## 2. Демонстрація: прогноз прогресування діабету

Набір `load_diabetes` з `scikit-learn`: 442 пацієнти, 10 медичних показників (вік, індекс маси
тіла, тиск, 6 аналізів крові); ціль — показник прогресування хвороби через рік.

Щоб показати проблему шуму, додамо 40 **випадкових** колонок, які ніяк не пов'язані з ціллю, —
так буває, коли в датасет «про всяк випадок» додають усе підряд.

In [ ]:
# Підготовка: бібліотеки та функція завантаження даних (працює в Colab і локально)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
DATA_URL = ""  # необов'язково: адреса теки з CSV (наприклад, raw-посилання на GitHub)


def load_csv(name):
    """Читає CSV з поточної теки, з теки data/, з DATA_URL або просить завантажити файл у Colab."""
    for path in (name, os.path.join("data", name)):
        if os.path.exists(path):
            return pd.read_csv(path)
    if DATA_URL:
        return pd.read_csv(DATA_URL.rstrip("/") + "/" + name)
    from google.colab import files
    print(f"Натисніть «Choose files» і виберіть файл {name}")
    files.upload()
    return pd.read_csv(name)

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import mean_absolute_error

diab = load_diabetes(as_frame=True)
X = diab.data.copy()
rng = np.random.default_rng(42)
for i in range(40):
    X[f"noise_{i}"] = rng.normal(scale=0.0476, size=len(X))  # той самий масштаб, що й у справжніх ознак
y = diab.target
print(X.shape)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

### Крок 1. Модель на всіх ознаках

In [ ]:
def test_mae(cols):
    m = LinearRegression().fit(X_tr[cols], y_tr)
    return mean_absolute_error(y_te, m.predict(X_te[cols]))

all_cols = list(X.columns)
real_cols = list(diab.data.columns)
print(f"Усі 50 ознак:        MAE = {test_mae(all_cols):.1f}")
print(f"Лише 10 справжніх:   MAE = {test_mae(real_cols):.1f}")

Шумові колонки погіршили прогноз: модель «підлаштувалася» під випадкові збіги в train.
У реальності ми не знаємо, які колонки шумові. Подивимося, чи знайдуть їх методи відбору.

### Крок 2. Кореляція з ціллю (на train)

In [ ]:
corr = X_tr.corrwith(y_tr).abs().sort_values(ascending=False)
corr.head(12).round(3)

Справжні показники (`bmi`, `s5`, `bp`, ...) — нагорі, але деякі шумові колонки теж мають
кореляцію 0.1–0.15 через випадковість. Кореляція дивиться на кожну ознаку окремо і лише на лінійний
зв'язок — це швидкий, але грубий фільтр.

### Крок 3. SelectKBest — залишити k найкращих

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

for k in [3, 5, 8, 10, 15, 30, 50]:
    selector = SelectKBest(f_regression, k=k).fit(X_tr, y_tr)
    cols = list(X_tr.columns[selector.get_support()])
    noise_in = sum(c.startswith("noise") for c in cols)
    print(f"k={k:2d}: MAE = {test_mae(cols):.1f}, шумових серед обраних: {noise_in}")

Зверніть увагу: **найкраще k обираємо не за test**, інакше test перестає бути чесною перевіркою.
Правильний спосіб — крос-валідація на train (заняття 10). Тут test використовуємо лише для
демонстрації, як k впливає на якість.

### Крок 4. Lasso — модель, яка сама обнуляє зайве

`Lasso` — лінійна регресія зі «штрафом» за великі коефіцієнти. Непотрібні ознаки отримують
коефіцієнт рівно 0. `LassoCV` сам підбирає силу штрафу на train.

In [ ]:
lasso = LassoCV(cv=5, random_state=42).fit(X_tr, y_tr)
coef = pd.Series(lasso.coef_, index=X.columns)
kept = list(coef[coef != 0].index)
print("Lasso залишив:", kept)
print(f"MAE моделі Lasso: {mean_absolute_error(y_te, lasso.predict(X_te)):.1f}")
print(f"MAE звичайної регресії на ознаках, які залишив Lasso: {test_mae(kept):.1f}")

Lasso відкинув більшість шумових колонок, і якість майже як у моделі на справжніх ознаках. Але кілька
шумових ознак лишилися з малим коефіцієнтом, а деякі справжні він відкинув —
жоден автоматичний метод не ідеальний, тому фінальне рішення поєднує метод і здоровий глузд.

### Крок 5. Мультиколінеарність: ознаки-«близнюки»

Якщо дві ознаки майже однакові (наприклад, пробіг у км і в милях), лінійна модель не може
вирішити, якій з них «віддати» вплив: коефіцієнти стають нестабільними й незрозумілими. Шукаємо
пари з кореляцією > 0.9 і лишаємо одну з пари.

In [ ]:
import seaborn as sns

plt.figure(figsize=(7, 6))
sns.heatmap(diab.data.corr().round(2), annot=True, cmap="coolwarm", center=0, fmt=".1f")
plt.title("Кореляції між медичними показниками")
plt.show()

`s1` і `s2` (два показники холестерину) корелюють приблизно на 0.9 — це «близнюки». Для прогнозу
це не критично, але коефіцієнти такої пари не варто пояснювати бізнесу окремо.

---
## 3. Практика: ціна вживаного авто

Сайт оголошень хоче підказувати продавцям ринкову ціну авто (`price_usd`). Датасет зібрали з
усього, що є в базі оголошень, — у ньому є зайве.

### Завдання 1. Ручні перевірки

Завантажте `used_cars.csv`. Знайдіть і поясніть у коментарях:
1. колонки-константи (`nunique() == 1`);
2. ідентифікатори;
3. дублікати інших колонок (кореляція > 0.99);
4. витік цілі.

Видаліть їх і запишіть список колонок, що лишилися.

In [ ]:
cars = load_csv("used_cars.csv")
# currency - константа; car_id - ідентифікатор; mileage_miles - той самий пробіг;
# price_uah - та сама ціна в гривнях (витік цілі)
# Ваш код тут

### Завдання 2. Базова модель на всіх ознаках

1. Розділіть на train/test (25 %, `random_state=42`).
2. Зберіть `Pipeline`: `OneHotEncoder(handle_unknown="ignore")` для текстових колонок, числові —
   без змін (`"passthrough"`), модель `LinearRegression`.
3. Порахуйте MAE на тесті в доларах.

In [ ]:
# Ваш код тут

### Завдання 3. Кореляції та підозрілі ознаки

1. Порахуйте кореляцію числових ознак з ціною **на train**. Які ознаки майже не пов'язані з ціною?
2. Подивіться на середню ціну за `color` і `region` (`groupby`). Чи відрізняються вони суттєво?

In [ ]:
# listing_views, photo_count майже не корелюють з ціною; колір і регіон майже не змінюють середню ціну.
# Ваш код тут

### Завдання 4. Модель без зайвого

Приберіть ознаки, які у завданні 3 виявилися непов'язаними з ціною, і навчіть ту саму модель.
Порівняйте MAE. Що виграв бізнес, навіть якщо MAE змінилась мало?

In [ ]:
# Якість майже та сама, але модель простіша: продавцю не треба вводити колір і регіон,
# а пояснювати ціну легше.
# Ваш код тут

### Завдання 5. Перевірте себе Lasso

Навчіть `LassoCV(cv=5, random_state=42)` на всіх ознаках після one-hot кодування
(числові ознаки перед цим масштабуйте `StandardScaler`, бо штраф Lasso залежить від масштабу).
Які ознаки Lasso обнулив? Чи збігається це з вашим ручним відбором?

In [ ]:
# Ваш код тут

---
## Підсумок

- Спершу **ручні перевірки**: константи, ідентифікатори, дублікати, витоки.
- Потім **фільтри** (кореляція, `SelectKBest`) і **модельні методи** (`Lasso`).
- Відбір робимо на train; рішення перевіряємо на test (а найкраще — крос-валідацією).
- Простіша модель з тією самою якістю — це виграш для бізнесу.